# Household master file, HIES 2024-25

This notebook builds **one file with one row per household** (30,123 households) from the 26 raw HIES 2024-25 files.

**Unit of analysis: the household.** Every household has its household weight (`weight`). Weighted, the file represents **40.1 million households**. Use `weight` for "% of households" and "number of households". `pop_weight` (= weight × household size) is only for stating how many people that is.

| Kind of raw file | How it enters the master file |
|---|---|
| **Household files**: `section_info`, housing (5M1), water and sanitation (5M2), waste (5M3), food insecurity (5M4) | Copied as they are: every question keeps its original column name and its text answer, from `CSV data/` |
| **Person files**: roster (1A), work and income (1B), education (2A/2B), ICT (2C) | Summarised per household: counts of members, the head's characteristics, household totals |
| **Item files**: spending (6A), durables (7), transfers (8A), property (9A), savings and loans (9B), farm (10A/10B), business (11) | Added up per household, converted to Rs per month |
| Health and women's sections (3A-3C, 4A-4E) | Not included: they describe children and women, not households. They belong in a person-level file. |

**Outputs** (in `outputs/`):
- `household_master.csv`: the master file
- `household_master_codebook.csv`: one row per column, with where it comes from and what it means

Spending and income use **exactly the same definitions as Experiment 1** (`../1_social_classes/`), so the averages match the numbers reported there and in the PBS report.

In [1]:
import os
import numpy as np
import pandas as pd
from IPython.display import display

RAW = "../../HIES 2024_2025/Stata data/"      # numeric codes, for calculations
CSV = "../../HIES 2024_2025/CSV data/"        # same data with text answers and corrected labels
OUT = "outputs/"
os.makedirs(OUT, exist_ok=True)
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_rows", 400)

def read(name, **kw):
    d = pd.read_stata(RAW + name, convert_categoricals=False, **kw)
    if "hhcode" in d:
        d["hhcode"] = d.hhcode.round().astype("int64")
    return d

def wmean(x, w):
    return float(np.average(x, weights=w))

pos = lambda s: s.fillna(0).clip(lower=0)      # money columns: missing = 0, no negatives

DESCRIBE = {}    # column -> (group, source, description); used to write the codebook at the end
def describe(group, source, **cols):
    for c, d in cols.items():
        DESCRIBE[c] = (group, source, d)

## 1. Identifiers, location and weight

- `hhcode` is the household ID (`prcode × 1000 + hhno`), used to join any raw file to this one.
- **Islamabad** is coded as Punjab (`province = 2`) in the raw files. Here `province` is taken from the first digit of `prcode`, so Islamabad is shown separately. `province_raw` keeps the original coding, which is what the PBS report tables use.
- The weight comes from `weight.dta`: one weight per PSU, the same for every household in it.

In [2]:
info = read("section_info.dta")
weight = read("weight.dta")
PROV = {1: "kp", 2: "punjab", 3: "sindh", 4: "balochistan", 6: "islamabad"}

m = pd.DataFrame({
    "hhcode": info.hhcode,
    "prcode": info.prcode.astype("int64"),
    "hhno": info.hhno.astype("int64"),
})
m["province"] = (m.prcode // 10_000_000).map(PROV)
m["province_raw"] = info.province.map({1: "kp", 2: "punjab", 3: "sindh", 4: "balochistan"})
m["region"] = info.region.map({1: "rural", 2: "urban"})
m["stratum"] = m.prcode // 100_000
m["quarter"] = (m.prcode // 1000) % 10
m["interview_date"] = pd.to_datetime(dict(year=info.enum_year, month=info.enum_month, day=info.enum_day), errors="coerce")
m = m.merge(weight[["prcode", "weight"]].astype({"prcode": "int64"}), on="prcode", how="left")
assert m.hhcode.is_unique and m.weight.notna().all() and len(m) == 30_123

describe("1 identifiers", "section_info, weight",
         hhcode="Household ID = prcode × 1000 + hhno. Use it to join any raw HIES file.",
         prcode="PSU code (8 digits): province, stratum, urban/rural, quarter, PSU serial",
         hhno="Household number inside the PSU",
         province="Province from the first digit of prcode; Islamabad shown separately",
         province_raw="Province as coded in the raw files (Islamabad counted in Punjab, as in PBS tables)",
         region="rural / urban",
         stratum="Sampling stratum = first 3 digits of prcode (158 strata)",
         quarter="Survey quarter (1-4), 5th digit of prcode",
         interview_date="Date of interview (enum_day/month/year)",
         weight="HOUSEHOLD WEIGHT: number of Pakistani households this household represents. Use for all % of households.")

## 2. Household members (roster, Section 1A)

A **household member** is a roster row with `s1aq10 = 1`; 556 people listed in the roster are not members and are not counted. The **head** is the member with relation `s1aq02 = 1`.

In [3]:
roster = read("plist_roster.dta")
mem = roster[roster.s1aq10 == 1].copy()
age = mem.s1aq51
g = mem.assign(
    male=mem.s1aq03 == 1, female=mem.s1aq03 == 2,
    a0_4=age.between(0, 4), a5_14=age.between(5, 14), a15_64=age.between(15, 64), a65=age >= 65,
    a5_16=age.between(5, 16), a10=age >= 10,
    women15_49=(mem.s1aq03 == 2) & age.between(15, 49),
    absent=mem.s1aq04 == 2,
).groupby("hhcode")
demo = pd.DataFrame({
    "hhsize": g.size(),
    "n_male": g.male.sum(), "n_female": g.female.sum(),
    "n_age_0_4": g.a0_4.sum(), "n_age_5_14": g.a5_14.sum(), "n_age_15_64": g.a15_64.sum(), "n_age_65plus": g.a65.sum(),
    "n_women_15_49": g.women15_49.sum(),
    "n_temporarily_absent": g.absent.sum(),
})
demo["dependency_ratio"] = (demo.n_age_0_4 + demo.n_age_5_14 + demo.n_age_65plus) / demo.n_age_15_64.replace(0, np.nan)
demo["n_non_members_listed"] = roster[roster.s1aq10 != 1].groupby("hhcode").size()
demo["n_non_members_listed"] = demo.n_non_members_listed.fillna(0).astype(int)

head = mem[mem.s1aq02 == 1].drop_duplicates("hhcode").set_index("hhcode")
MARITAL = {1: "never married", 2: "currently married", 3: "widowed", 4: "divorced", 5: "separated", 6: "nikah, rukhsati not yet"}
demo["head_idc"] = head.idc
demo["head_sex"] = head.s1aq03.map({1: "male", 2: "female"})
demo["head_age"] = head.s1aq51
demo["head_marital_status"] = head.s1aq06.map(MARITAL)

m = m.merge(demo.reset_index(), on="hhcode", how="left")
m["pop_weight"] = m.weight * m.hhsize
print(f"Households without a head among members: {m.head_sex.isna().sum()}")

describe("2 members", "plist_roster (members: s1aq10 = 1)",
         hhsize="Number of household members",
         n_male="Male members", n_female="Female members",
         n_age_0_4="Members aged 0-4", n_age_5_14="Members aged 5-14", n_age_15_64="Members aged 15-64",
         n_age_65plus="Members aged 65+", n_women_15_49="Female members aged 15-49",
         n_temporarily_absent="Members temporarily absent at the interview (s1aq04 = 2)",
         dependency_ratio="(members aged 0-14 + 65+) ÷ members aged 15-64; blank if no member is 15-64",
         n_non_members_listed="People listed in the roster who are NOT household members (s1aq10 = 2); not counted anywhere else",
         head_idc="Person ID (idc) of the household head, to join person files",
         head_sex="Sex of the household head", head_age="Age of the household head (years)",
         head_marital_status="Marital status of the household head",
         pop_weight="weight × hhsize. ONLY for stating a population total; do not use for household shares.")
describe("1 identifiers", "weight × roster", pop_weight=DESCRIBE.pop("pop_weight")[2])

Households without a head among members: 4


## 3. Education (Sections 2A, 2B)

- **Literate** = can read **and** write a simple statement with understanding (`s2aq01 = 1` and `s2aq02 = 1`), counted for members aged 10+.
- The head's education is the highest grade completed (`s2bq05`), grouped into levels. Someone still studying has their current grade in `s2bq10` instead, so for them we use the grade below it.

In [4]:
edu = read("sec_2ab_education.dta").merge(mem[["hhcode", "idc", "s1aq51"]], on=["hhcode", "idc"])
edu["literate"] = (edu.s2aq01 == 1) & (edu.s2aq02 == 1)
# Grade codes -> years of schooling (codes 25-27 are pre-school, 28 'other' is unknown)
YEARS = {**{k: k for k in range(1, 11)}, 11: 12, 12: 12, 13: 14, 14: 15, 15: 16, 16: 16, 17: 16, 18: 16,
         19: 16, 20: 16, 21: 16, 22: 18, 23: 20, 24: 18, 25: 0, 26: 0, 27: 0}
completed = edu.s2bq05.map(YEARS)
current = (edu.s2bq10.map(YEARS) - 1).clip(lower=0)
edu["years"] = np.where(edu.s2bq01 == 1, 0, np.where(edu.s2bq01 == 3, current, completed))

def level(y):
    if pd.isna(y): return np.nan
    return ("none" if y == 0 else "primary (1-5)" if y <= 5 else "middle (6-8)" if y <= 8
            else "matric (9-10)" if y <= 10 else "intermediate (11-12)" if y <= 12 else "graduate or higher")

ed = pd.DataFrame({
    "n_literate_10plus": edu[edu.literate & (edu.s1aq51 >= 10)].groupby("hhcode").size(),
    "n_age_5_16": edu[edu.s1aq51.between(5, 16)].groupby("hhcode").size(),
    "n_age_5_16_in_school": edu[edu.s1aq51.between(5, 16) & (edu.s2bq01 == 3)].groupby("hhcode").size(),
    "max_years_schooling": edu.groupby("hhcode").years.max(),
}).fillna({"n_literate_10plus": 0, "n_age_5_16": 0, "n_age_5_16_in_school": 0})
h = edu.set_index(["hhcode", "idc"])
key = pd.MultiIndex.from_arrays([m.hhcode, m.head_idc.fillna(-1).astype(int)])
m["head_literate"] = h.literate.reindex(key).map({True: "yes", False: "no"}).to_numpy()
m["head_years_schooling"] = h.years.reindex(key).to_numpy()
m["head_education"] = m.head_years_schooling.map(level)
m = m.merge(ed.reset_index(), on="hhcode", how="left")
m["n_age_10plus"] = mem[mem.s1aq51 >= 10].groupby("hhcode").size().reindex(m.hhcode).fillna(0).to_numpy()
m["share_literate_10plus"] = m.n_literate_10plus / m.n_age_10plus.replace(0, np.nan)

describe("3 education", "sec_2ab_education (members only)",
         head_literate="Head can read and write a simple statement (s2aq01 = 1 and s2aq02 = 1)",
         head_years_schooling="Head's years of schooling, from the highest grade completed (s2bq05)",
         head_education="Head's education level: none / primary / middle / matric / intermediate / graduate or higher",
         n_literate_10plus="Members aged 10+ who can read and write",
         n_age_10plus="Members aged 10+",
         share_literate_10plus="n_literate_10plus ÷ n_age_10plus",
         n_age_5_16="Members aged 5-16 (with an education record)",
         n_age_5_16_in_school="Members aged 5-16 currently attending school (s2bq01 = 3)",
         max_years_schooling="Highest years of schooling of any member")

## 4. Work and cash income (Section 1B)

Cash income is defined as in Experiment 1. For each person aged 10+, the yearly total is:
- main job: last year's earnings `s1bq08`, or if missing, last month × months worked (`s1bq06 × s1bq07`);
- plus second job `s1bq10`, other work `s1bq11`, `s1bq12`, and pensions `s1bq13`.

The household total is divided by 12. An **earner** is anyone with income above 0.

In [5]:
emp = read("sec_1b_emp_income.dta")
months_worked = emp.s1bq07.fillna(12).clip(1, 12)
main_job_year = np.where(pos(emp.s1bq08) > 0, pos(emp.s1bq08), pos(emp.s1bq06) * months_worked)
emp["income_year"] = main_job_year + pos(emp.s1bq10) + pos(emp.s1bq11) + pos(emp.s1bq12) + pos(emp.s1bq13)
emp["is_earner"] = emp.income_year > 0
w = emp.groupby("hhcode")
work = pd.DataFrame({
    "n_working": emp[emp.s1bq01 == 1].groupby("hhcode").size(),
    "n_earners": w.is_earner.sum(),
    "income_cash_month": w.income_year.sum() / 12,
    "income_main_earner_month": w.income_year.max() / 12,
}).fillna(0)
STATUS = {1: "employer (<10 workers)", 2: "employer (10+ workers)", 3: "self-employed non-agri", 4: "paid employee (monthly)",
          5: "paid employee (daily wage)", 6: "unpaid family worker", 7: "owner cultivator", 8: "share cropper",
          9: "contract cultivator", 10: "livestock only", 11: "student", 12: "housekeeping", 13: "old / child",
          14: "landlord", 15: "disabled", 16: "pensioner / retired", 17: "looking for work", 18: "other"}
he = emp.set_index(["hhcode", "idc"])
m["head_working"] = he.s1bq01.reindex(key).map({1: "yes", 2: "no"}).to_numpy()
m["head_employment_status"] = he.s1bq02.reindex(key).map(STATUS).to_numpy()
m = m.merge(work.reset_index(), on="hhcode", how="left")
m[work.columns] = m[work.columns].fillna(0)

describe("4 work", "sec_1b_emp_income (age 10+)",
         head_working="Head has paid work, unpaid family work, a business or farm (s1bq01)",
         head_employment_status="Head's employment status (s1bq02)",
         n_working="Members aged 10+ who work (s1bq01 = 1)",
         n_earners="Members with cash income from work or pension above 0",
         income_cash_month="Household CASH income from work and pensions, Rs per month (Section 1B; as in Experiment 1)",
         income_main_earner_month="Cash income of the household's highest earner, Rs per month")

## 5. Phones, internet and bank accounts (Section 2C)

In [6]:
ict = read("sec_02c_ict.dta").merge(mem[["hhcode", "idc"]], on=["hhcode", "idc"])
c = ict.groupby("hhcode")
ictd = pd.DataFrame({
    "n_own_mobile": ict[ict.s2cq02.isin([1, 2, 3])].groupby("hhcode").size(),
    "n_own_smartphone": ict[ict.s2cq02.isin([2, 3])].groupby("hhcode").size(),
    "n_used_internet": ict[ict.s2cq05 == 1].groupby("hhcode").size(),
    "n_bank_or_digital_account": ict[ict.s2cq15.isin([1, 2, 3])].groupby("hhcode").size(),
}).fillna(0)
m = m.merge(ictd.reset_index(), on="hhcode", how="left")
m[ictd.columns] = m[ictd.columns].fillna(0)
describe("5 ict", "sec_02c_ict (members)",
         n_own_mobile="Members who own a mobile or smartphone (s2cq02 = 1, 2, 3)",
         n_own_smartphone="Members who own a smartphone (s2cq02 = 2, 3)",
         n_used_internet="Members who used the internet in the last 3 months (s2cq05 = 1)",
         n_bank_or_digital_account="Members with a bank or digital (Easypaisa, JazzCash) account (s2cq15 = 1, 2, 3)")

## 6. Household questionnaires copied as they are (Section 5 and interview status)

These files already have one row per household. Every question is copied with its **original column name** and its **text answer** from `CSV data/`, where the two PBS label errors are corrected. The meaning of each column is in the codebook.

The **FIES score** counts "yes" answers to the 8 food-insecurity questions (0-8). "Don't know" and "refused" count as not yes.

In [7]:
ID_COLS = {"prcode", "hhno", "province", "region", "quarter", "quater", "hhcode"}
cb_raw = pd.read_csv(CSV + "_codebook.csv")
SECTIONS = [("section_info", "6 interview"), ("sec_05m1_housingchar", "7 housing (5M1)"),
            ("sec_05m2_watersanitation", "8 water & sanitation (5M2)"),
            ("sec_05m3_wastemanagment", "9 waste (5M3)"), ("sec_05m4_fies", "10 food insecurity (5M4)")]
for f, grp in SECTIONS:
    d = pd.read_csv(CSV + f + ".csv", low_memory=False)
    keep = [c for c in d.columns if c not in ID_COLS and c not in ("enum_day", "enum_month", "enum_year")]
    renamed = {c: ("fies_" + c if f == "sec_05m4_fies" else c) for c in keep}
    m = m.merge(d[["hhcode"] + keep].rename(columns=renamed), on="hhcode", how="left")
    lab = cb_raw[cb_raw.file == f].set_index("column")
    for c in keep:
        desc = lab.description.get(c)
        if not isinstance(desc, str):
            desc = f"Text written for 'other (specify)', see {c.rstrip('t').rstrip('o')}" if c.endswith("t") else c
        DESCRIBE[renamed[c]] = (grp, f, desc)

DESCRIBE["fstatus"] = ("6 interview", "section_info", "Interview status (all households: completed)")
m["fies_score"] = sum((m[f"fies_q{i}"] == "yes").astype(int) for i in range(1, 9))
describe("10 food insecurity (5M4)", "sec_05m4_fies",
         fies_score="Number of 'yes' answers to the 8 FIES questions (0 = food secure ... 8 = most severe)")
print(m.fstatus.value_counts())

fstatus
completed    30123
Name: count, dtype: int64


## 7. Spending (Section 6A)

Exactly as in Experiment 1 (see `../1_social_classes/HOW_CLASSES_ARE_DEFINED.md`):
- item value = `v1` bought + `v2` wages in kind + `v3` own production + `v4` gifts; page-total rows (`itc` < 10000) are dropped;
- converted to one month: part A (14 days) × 365/12/14; parts B and C × 1; parts D and E (12 months) ÷ 12;
- grouped into the 12 COICOP divisions by the first digits of `itc`.

**Income in kind** is the part of spending never paid in cash (`v2` + `v3` + `v4`). `exp_imputed_rent` is the rental value of an owner-occupied home (item 42101), which is part of it.

In [8]:
raw6 = read("sec_6a_consum_exp.dta", columns=["hhcode", "itc", "v1", "v2", "v3", "v4"])
raw6["value"] = raw6[["v1", "v2", "v3", "v4"]].fillna(0).sum(axis=1)

def part_of(code):
    group = code // 100
    if group in (112, 113, 114, 116, 117, 122) or 11801 <= code <= 11810 or 11901 <= code <= 11908:
        return "A"
    if group in (111, 115, 121, 561, 1211, 1212, 1213) or 11909 <= code <= 11912 or 110000 <= code < 111000:
        return "B"
    if code // 1000 in (22, 45) or group in (441, 442):
        return "C"
    if (code // 1000 == 51 or group in (522, 911) or 53103 <= code <= 53107 or 54104 <= code <= 54107
            or code in (55101, 55201, 71101, 82101, 123102, 123103)):
        return "E"
    return "D"

items = raw6[(raw6.itc >= 10000) & (raw6.value > 0)].copy()
items["part"] = items.itc.map({c: part_of(c) for c in items.itc.unique()})
PER_MONTH = {"A": 365 / 12 / 14, "B": 1, "C": 1, "D": 1 / 12, "E": 1 / 12}
factor = items.part.map(PER_MONTH)
for v in ["v1", "v2", "v3", "v4"]:
    items[v] = items[v].fillna(0).clip(lower=0) * factor
items["monthly"] = items.value * factor

COICOP = {1: "food", 2: "tobacco", 3: "clothing", 4: "housing_utilities", 5: "furnishings", 6: "health",
          7: "transport", 8: "communication", 9: "recreation", 10: "education", 11: "restaurants_hotels",
          12: "personal_care_misc"}
items["division"] = items.itc // 10000
spend = items.pivot_table(index="hhcode", columns="division", values="monthly", aggfunc="sum").fillna(0)
spend.columns = ["exp_" + COICOP[c] for c in spend.columns]
EXP_COLS = list(spend.columns)
s = items.groupby("hhcode")
spend["exp_total"] = spend[EXP_COLS].sum(axis=1)
spend["exp_bought"] = s.v1.sum()
spend["inkind_wages"] = s.v2.sum()
spend["inkind_own_production"] = s.v3.sum()
spend["inkind_gifts"] = s.v4.sum()
spend["exp_imputed_rent"] = items[items.itc == 42101].groupby("hhcode").monthly.sum()
spend = spend.fillna(0)
m = m.merge(spend.reset_index(), on="hhcode", how="left")
m["exp_per_person"] = m.exp_total / m.hhsize
m["exp_food_share"] = m.exp_food / m.exp_total

# Spending quintile, by households: sort by spending per person, cut the cumulative household weight at 20/40/60/80%
order = m.sort_values("exp_per_person")
cum = order.weight.cumsum() / order.weight.sum()
m.loc[order.index, "quintile_hh"] = np.minimum(np.ceil(cum * 5), 5).astype(int)
m["quintile_hh"] = m.quintile_hh.astype(int)

NICE = {"food": "food and non-alcoholic drinks", "housing_utilities": "rent (incl. owned home), water, electricity, gas, fuel",
        "furnishings": "furnishings, appliances, household goods", "restaurants_hotels": "restaurants and hotels",
        "personal_care_misc": "personal care and miscellaneous"}
describe("11 spending (6A)", "sec_6a_consum_exp",
         exp_total="TOTAL household consumption spending, Rs per month (bought + in kind; as in Experiment 1)",
         exp_per_person="exp_total ÷ hhsize, Rs per person per month (used to rank households)",
         exp_food_share="exp_food ÷ exp_total",
         exp_bought="Part of exp_total paid in cash (v1), Rs per month",
         inkind_wages="Consumption received as wages in kind (v2), Rs per month",
         inkind_own_production="Consumption from own production (v3), incl. owned-home rent, Rs per month",
         inkind_gifts="Consumption received as gifts or assistance (v4), Rs per month",
         exp_imputed_rent="Rental value of an owner-occupied home (item 42101), Rs per month; included in exp_housing_utilities",
         quintile_hh="Spending quintile (1 = poorest), 20% of HOUSEHOLDS each, by exp_per_person. PBS report quintiles are 20% of POPULATION instead.")
for c in EXP_COLS:
    k = c[4:]
    describe("11 spending (6A)", "sec_6a_consum_exp", **{c: f"Spending on {NICE.get(k, k.replace('_', ' '))}, Rs per month (COICOP division)"})

## 8. Durable goods (Section 7)

The number of each of 35 items the household **owns now** (`s7m_b`). 21 households have no Section 7 record; they are set to 0.

In [9]:
dur = read("sec_7.dta")
DUR = {701: "radio", 702: "television", 703: "lcd_led", 704: "refrigerator", 705: "freezer", 706: "washing_machine",
       707: "dryer", 708: "air_conditioner", 709: "air_cooler", 710: "fan", 711: "stove", 712: "cooking_range",
       713: "microwave", 714: "sewing_machine", 715: "knitting_machine", 716: "iron", 717: "water_filter",
       718: "donkey_pump", 719: "turbine", 720: "chair", 721: "table", 722: "ups", 723: "generator",
       724: "solar_panel", 725: "heater", 726: "geyser", 727: "bicycle", 728: "motorcycle", 729: "scooter",
       730: "car", 731: "van_truck_bus", 732: "motor_boat", 733: "tractor_trolley", 734: "watch", 735: "animal_cart"}
d7 = dur[dur.itc.isin(DUR)].pivot_table(index="hhcode", columns="itc", values="s7m_b", aggfunc="sum")
d7.columns = ["own_" + DUR[c] for c in d7.columns]
d7 = d7.reindex(columns=["own_" + v for v in DUR.values()])
m = m.merge(d7.reset_index(), on="hhcode", how="left")
m[d7.columns] = m[d7.columns].fillna(0)
for k, v in DUR.items():
    describe("12 durables (7)", "sec_7", **{"own_" + v: f"Number of {v.replace('_', ' ')} owned now (item {k}, s7m_b)"})

## 9. Transfers, property, savings, farm and business (Sections 8A, 9A, 9B, 10A, 10B, 11)

Yearly amounts are divided by 12, so **every money flow is in Rs per month**. Stocks (savings, gold, loans owed) are values at the interview date.

`transfers` and `property_income` use **the same codes as Experiment 1**, and `income_full` adds them to cash income and income in kind:

> income_full = income_cash_month + transfers + property_income + income in kind (`inkind_wages` + `inkind_own_production` + `inkind_gifts`)

In [10]:
def by_hh(df, codes, col, per_month=True):
    d = df[df.code.isin(codes)]
    out = pos(d[col]).groupby(d.hhcode).sum()
    return out / 12 if per_month else out

t8, p9, f9 = read("sec_8a.dta"), read("sec_9a.dta"), read("sec_9b.dta")
a10, l10, b11 = read("sec_10a_agrisheet.dta"), read("sec_10b_livestock.dta"), read("sec_11ab_nonagri.dta")

money = pd.DataFrame({
    # Section 8A transfers received (yearly -> monthly)
    "tr_remit_domestic": by_hh(t8, [802], "c02"),
    "tr_remit_abroad": by_hh(t8, [805, 806, 807, 808], "c02"),
    "tr_bisp_govt": by_hh(t8, [820, 821], "c02"),
    "tr_assistance": by_hh(t8, [812, 813], "c02"),
    "tr_boarders_gifts_sold": by_hh(t8, [817, 819], "c02"),
    "tr_zakat_usher": by_hh(t8, [811], "c02"),
    "tr_received_total_8a": by_hh(t8, [800], "c02"),
    "tr_paid_total_8a": by_hh(t8, [850], "c02"),
    # rent: property (9A), business assets rented out (11), farm equipment rented out (10B code 197)
    "rent_received": by_hh(p9, [901, 902, 903, 904], "s9aq04")
        .add(by_hh(b11, [288, 289], "s11a"), fill_value=0)
        .add(by_hh(a10, [197], "s10c1"), fill_value=0),
    "profit_dividends": by_hh(f9, [953, 960, 970], "c02"),
    # stocks (not per month)
    "savings_now": by_hh(f9, [951], "c02", per_month=False),
    "gold_jewellery_value": by_hh(f9, [955], "c02", per_month=False),
    "loans_owed_now": by_hh(f9, [963], "c02", per_month=False),
    "farm_crop_value_month": by_hh(a10, [135], "s10c3"),
}).reindex(m.hhcode).fillna(0)
money["transfers"] = money[["tr_remit_domestic", "tr_remit_abroad", "tr_bisp_govt", "tr_assistance", "tr_boarders_gifts_sold"]].sum(axis=1)
money["property_income"] = money.rent_received + money.profit_dividends
m = pd.concat([m.set_index("hhcode"), money], axis=1).reset_index()

land = p9[p9.code == 901].set_index("hhcode")
home = p9[p9.code == 903].set_index("hhcode")
m["owns_agri_land"] = land.s9aq02a.reindex(m.hhcode).map({1: "yes", 2: "no"}).fillna("no").to_numpy()
m["agri_land_acres"] = pos(land.s9aq02b.reindex(m.hhcode)).to_numpy()
m["owns_residential_property"] = home.s9aq02a.reindex(m.hhcode).map({1: "yes", 2: "no"}).fillna("no").to_numpy()
m["farm_household"] = np.where(m.hhcode.isin(a10.hhcode), "yes", "no")
m["livestock_household"] = np.where(m.hhcode.isin(l10.hhcode), "yes", "no")
m["n_nonfarm_businesses"] = b11.drop_duplicates(["hhcode", "idc", "sheetno"]).groupby("hhcode").size().reindex(m.hhcode).fillna(0).to_numpy()

m["income_in_kind"] = m.inkind_wages + m.inkind_own_production + m.inkind_gifts
m["income_full"] = m.income_cash_month + m.transfers + m.property_income + m.income_in_kind
m["income_full_per_person"] = m.income_full / m.hhsize

describe("13 transfers (8A)", "sec_8a (yearly ÷ 12)",
         tr_remit_domestic="Remittances received from within Pakistan (802), Rs per month",
         tr_remit_abroad="Remittances received from abroad: bank + hundi + mobile + other (805-808), Rs per month",
         tr_bisp_govt="BISP/Ehsaas and other government assistance (820, 821), Rs per month",
         tr_assistance="Assistance received from public and private sector (812, 813), Rs per month",
         tr_boarders_gifts_sold="Receipts from boarders, and gifts received in kind but sold (817, 819), Rs per month",
         tr_zakat_usher="Zakat/usher received in cash (811), Rs per month. NOT in 'transfers' (kept as in Experiment 1)",
         tr_received_total_8a="Section 8A part A total as recorded (800): all receipts incl. committees, insurance, inheritance, Rs per month",
         tr_paid_total_8a="Section 8A part B total as recorded (850): all transfers paid out, Rs per month",
         transfers="Transfer income used in income_full = 802 + 805-808 + 820-821 + 812-813 + 817 + 819, Rs per month")
describe("14 property & savings (9A, 9B)", "sec_9a, sec_9b, sec_10a, sec_11",
         rent_received="Net rent received: land/buildings (9A) + business assets (11: 288, 289) + farm equipment (10: 197), Rs per month",
         profit_dividends="Profit on savings, dividends, profit on loans given (9B: 953, 960, 970), Rs per month",
         property_income="rent_received + profit_dividends, Rs per month",
         savings_now="Total net savings at present (9B: 951), Rs",
         gold_jewellery_value="Value of gold, silver and jewellery owned (9B: 955), Rs",
         loans_owed_now="Loans currently owed by the household (9B: 963), Rs",
         owns_agri_land="Household owns all or part of agricultural land now (9A: 901, s9aq02a)",
         agri_land_acres="Acres of agricultural land (9A: 901, s9aq02b)",
         owns_residential_property="Household owns a residential building now (9A: 903, s9aq02a)")
describe("15 farm & business (10, 11)", "sec_10a, sec_10b, sec_11ab",
         farm_household="Household has a crop farming sheet (Section 10A)",
         livestock_household="Household has a livestock sheet (Section 10B)",
         farm_crop_value_month="Value of total crop production (10A: 135, s10c3) ÷ 12, Rs per month. Not added to income_full: farm cash income is already in Section 1B.",
         n_nonfarm_businesses="Number of non-farm businesses (Section 11 sheets)")
describe("16 income totals", "derived",
         income_in_kind="inkind_wages + inkind_own_production + inkind_gifts, Rs per month",
         income_full="FULL household income = income_cash_month + transfers + property_income + income_in_kind, Rs per month (as in Experiment 1)",
         income_full_per_person="income_full ÷ hhsize, Rs per person per month")

## 10. Checks

The master file must reproduce the numbers already reported:
- Experiment 1 and the PBS HIES 2024-25 report (average household size 5.98, spending Rs 81,351, full income Rs 84,057);
- the household-level file `household_classes.csv` from Experiment 1, household by household.

In [11]:
assert len(m) == 30_123 and m.hhcode.is_unique
W = m.weight
checks = pd.DataFrame([
    ("Households (sum of weights)", W.sum(), "40,115,424 (Experiment 1)"),
    ("Population (sum of weight × hhsize)", m.pop_weight.sum(), "239,823,498 (Experiment 1)"),
    ("Average household size", m.pop_weight.sum() / W.sum(), "5.98 (PBS report Table 3.2)"),
    ("Average household size, urban", m[m.region == "urban"].pop_weight.sum() / m[m.region == "urban"].weight.sum(), "5.69 (PBS report)"),
    ("Average household size, rural", m[m.region == "rural"].pop_weight.sum() / m[m.region == "rural"].weight.sum(), "6.18 (PBS report)"),
    ("Spending per household per month (Rs)", wmean(m.exp_total, W), "81,351 (Experiment 1); PBS 79,150"),
    ("Full income per household per month (Rs)", wmean(m.income_full, W), "84,057 (Experiment 1); PBS 82,179"),
], columns=["check", "master file", "expected"])
display(checks)

e1 = pd.read_csv("../1_social_classes/outputs/household_classes.csv").set_index("hhcode")
mm = m.set_index("hhcode").loc[e1.index]
pairs = {"hhsize": "hhsize", "weight": "weight", "exp_total": "monthly_total", "income_cash_month": "income_month",
         "n_earners": "earners", "transfers": "transfers", "property_income": "property_income",
         "income_in_kind": "in_kind", "income_full": "full_income"}
same = {a: f"{(np.abs(mm[a] - e1[b]) <= 0.1 + 1e-6 * np.abs(e1[b])).mean() * 100:.2f}%" for a, b in pairs.items()}
print("Households where the master file equals Experiment 1's household_classes.csv:")
print(same)

,check,master file,expected
0,Households (sum of weights),"40,115,424.00","40,115,424 (Experiment 1)"
1,Population (sum of weight × hhsize),"239,823,497.65","239,823,498 (Experiment 1)"
2,Average household size,5.98,5.98 (PBS report Table 3.2)
3,"Average household size, urban",5.69,5.69 (PBS report)
4,"Average household size, rural",6.18,6.18 (PBS report)
5,Spending per household per month (Rs),"81,351.17","81,351 (Experiment 1); PBS 79,150"
6,Full income per household per month (Rs),"84,056.68","84,057 (Experiment 1); PBS 82,179"


Households where the master file equals Experiment 1's household_classes.csv:
{'hhsize': '100.00%', 'weight': '100.00%', 'exp_total': '100.00%', 'income_cash_month': '100.00%', 'n_earners': '100.00%', 'transfers': '100.00%', 'property_income': '100.00%', 'income_in_kind': '100.00%', 'income_full': '100.00%'}


## 11. Save the file and its codebook

In [12]:
FIRST = ["hhcode", "prcode", "hhno", "province", "province_raw", "region", "stratum", "quarter", "interview_date", "weight", "pop_weight"]
m = m[FIRST + [c for c in m.columns if c not in FIRST]]
m = m.drop(columns=["head_idc"]).assign(head_idc=m.head_idc)       # keep head_idc but move it after the head columns
m.round(2).to_csv(OUT + "household_master.csv", index=False, encoding="utf-8-sig")

missing = [c for c in m.columns if c not in DESCRIBE]
assert not missing, missing
codebook = pd.DataFrame([(c, *DESCRIBE[c]) for c in m.columns], columns=["column", "group", "source", "description"])
codebook["non_missing"] = m.notna().sum().to_numpy()
codebook["example"] = [m[c].dropna().iloc[0] if m[c].notna().any() else "" for c in m.columns]
codebook.to_csv(OUT + "household_master_codebook.csv", index=False, encoding="utf-8-sig")

size_mb = os.path.getsize(OUT + "household_master.csv") / 1e6
print(f"household_master.csv: {len(m):,} households × {m.shape[1]} columns, {size_mb:.1f} MB")
codebook.groupby("group", sort=False).size().rename("columns").to_frame()

household_master.csv: 30,123 households × 234 columns, 36.9 MB


,columns
group,
1 identifiers,11
2 members,15
3 education,9
4 work,6
5 ict,4
6 interview,1
7 housing (5M1),68
8 water & sanitation (5M2),24
9 waste (5M3),6


## 12. Comparison with well-known official statistics

**Benchmark: the 7th Population and Housing Census 2023** (PBS; reference date 1 March 2023). Like HIES, it covers the four provinces and Islamabad, not AJK or Gilgit-Baltistan. The census figures are typed in below from the PBS census releases.

HIES weights are household weights, so the **number of households** is the survey's own figure. **Population** is households × HIES household size.

In [13]:
CENSUS = {   # 7th Population and Housing Census 2023 (PBS)
    "population": 241.49e6, "households": 38.29e6, "hh_size": 6.30, "rural_pop_pct": 61.18,
    "pop_by_province": {"punjab": 127.69e6, "sindh": 55.69e6, "kp": 40.85e6, "balochistan": 14.89e6, "islamabad": 2.36e6},
    "rural_pct_by_province": {"punjab": 59, "sindh": 47, "kp": 85, "balochistan": 69, "islamabad": 53},  # rounded in the census releases
    "households_2017": 31.92e6, "pop_growth_2017_23": 2.55,
    "grid_electricity_lighting_pct": 84.03, "solar_lighting_pct": 7.74,
}
dates = m.interview_date.dropna().sort_values() if m.interview_date.dtype != object else pd.to_datetime(m.interview_date).dropna().sort_values()
mid = pd.Timestamp(np.median(dates.astype("int64")))
years = (mid - pd.Timestamp("2023-03-01")).days / 365.25
hh_growth = (CENSUS["households"] / CENSUS["households_2017"]) ** (1 / 6) - 1          # 2017 -> 2023, per year
print(f"HIES interviews: {dates.min():%d %b %Y} to {dates.max():%d %b %Y}; middle {mid:%d %b %Y}, {years:.2f} years after the census")
print(f"Household growth 2017-2023: {hh_growth * 100:.2f}% a year; population growth: {CENSUS['pop_growth_2017_23']}% a year")

rural = m.region == "rural"
rows = [
    ("Households (million)", W.sum() / 1e6, CENSUS["households"] / 1e6, CENSUS["households"] * (1 + hh_growth) ** years / 1e6),
    ("Population (million)", m.pop_weight.sum() / 1e6, CENSUS["population"] / 1e6, CENSUS["population"] * (1 + CENSUS["pop_growth_2017_23"] / 100) ** years / 1e6),
    ("Average household size", m.pop_weight.sum() / W.sum(), CENSUS["hh_size"], np.nan),
    ("% of population rural", m.pop_weight[rural].sum() / m.pop_weight.sum() * 100, CENSUS["rural_pop_pct"], np.nan),
    ("% of households rural", W[rural].sum() / W.sum() * 100, np.nan, np.nan),
    ("% of households lit by grid electricity (incl. grid + solar)", W[m.s5m1q10.str.startswith("electricity", na=False)].sum() / W.sum() * 100, CENSUS["grid_electricity_lighting_pct"], np.nan),
    ("% of households lit by solar only", W[m.s5m1q10 == "solar only"].sum() / W.sum() * 100, CENSUS["solar_lighting_pct"], np.nan),
]
national = pd.DataFrame(rows, columns=["indicator", "HIES 2024-25", "Census 2023", "Census 2023 projected to HIES mid-point"])
display(national.round(2))

prov = pd.DataFrame({
    "HIES households (M)": m.groupby("province").weight.sum() / 1e6,
    "HIES % of households": m.groupby("province").weight.sum() / W.sum() * 100,
    "HIES population (M)": m.groupby("province").pop_weight.sum() / 1e6,
    "Census population (M)": pd.Series(CENSUS["pop_by_province"]) / 1e6,
    "HIES % of population": m.groupby("province").pop_weight.sum() / m.pop_weight.sum() * 100,
    "Census % of population": pd.Series(CENSUS["pop_by_province"]) / sum(CENSUS["pop_by_province"].values()) * 100,
    "HIES % rural (population)": m[rural].groupby("province").pop_weight.sum() / m.groupby("province").pop_weight.sum() * 100,
    "Census % rural (population)": pd.Series(CENSUS["rural_pct_by_province"]),
    "HIES household size": m.groupby("province").pop_weight.sum() / m.groupby("province").weight.sum(),
}).loc[["punjab", "sindh", "kp", "balochistan", "islamabad"]]
prov.round(2)

HIES interviews: 01 Oct 2024 to 30 Jun 2025; middle 11 Feb 2025, 1.95 years after the census
Household growth 2017-2023: 3.08% a year; population growth: 2.55% a year


,indicator,HIES 2024-25,Census 2023,Census 2023 projected to HIES mid-point
0,Households (million),40.12,38.29,40.63
1,Population (million),239.82,241.49,253.66
2,Average household size,5.98,6.30,NaN
3,% of population rural,61.25,61.18,NaN
4,% of households rural,59.28,NaN,NaN
5,% of households lit by grid electricity (incl....,88.25,84.03,NaN
6,% of households lit by solar only,7.43,7.74,NaN


,HIES households (M),HIES % of households,HIES population (M),Census population (M),HIES % of population,Census % of population,HIES % rural (population),Census % rural (population),HIES household size
punjab,21.98,54.79,126.69,127.69,52.83,52.88,59.46,59,5.76
sindh,9.73,24.25,55.45,55.69,23.12,23.06,46.17,47,5.70
kp,5.94,14.82,40.61,40.85,16.93,16.92,85.13,85,6.83
balochistan,2.08,5.19,14.75,14.89,6.15,6.17,68.82,69,7.08
islamabad,0.38,0.95,2.33,2.36,0.97,0.98,53.21,53,6.10
